In [2]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import jax
import jax.numpy as jnp
import numpy as onp

In [3]:
output_shape = (3,)
fun = lambda x: x**2
alternative = lambda x: jnp.zeros(output_shape)

x = jnp.array([1.1, 2.22, 3.333])

jax.lax.cond(True, fun, alternative, x)

Array([ 1.21    ,  4.9284  , 11.108889], dtype=float64)

In [4]:
output_shape_2 = (1, 3)
alternative_output_2 = (0.0, jnp.zeros(3))

def fun2(x):
    return jnp.linalg.norm(x), 2 * x

def alternative2(x):
    return alternative_output_2

def conditional_eval(pred, x):
    return jax.lax.cond(pred, fun2, alternative2, x)

In [5]:
conditional_eval(True, x)

(Array(4.15298555, dtype=float64, weak_type=True),
 Array([2.2  , 4.44 , 6.666], dtype=float64))

In [6]:
# conditions = jnp.array([[True, True, False, False], [True, False, False, False], [True, True, True, True], [True, True, True, False]])

In [7]:
conditions = jnp.array([True, True, False, False])

In [8]:
xs = jnp.outer(jnp.linspace(1, len(conditions) + 1, 4), jnp.arange(1, 4))
xs

Array([[ 1.        ,  2.        ,  3.        ],
       [ 2.33333333,  4.66666667,  7.        ],
       [ 3.66666667,  7.33333333, 11.        ],
       [ 5.        , 10.        , 15.        ]], dtype=float64)

In [9]:
xs[0]

Array([1., 2., 3.], dtype=float64)

In [10]:
conditions.shape, xs.shape

((4,), (4, 3))

In [11]:
jax.vmap(conditional_eval, in_axes=(0, 0))(conditions, xs)

(Array([3.74165739, 8.7305339 , 0.        , 0.        ], dtype=float64),
 Array([[ 2.        ,  4.        ,  6.        ],
        [ 4.66666667,  9.33333333, 14.        ],
        [ 0.        ,  0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ]], dtype=float64))

In [12]:
jax.tree_map(lambda x: jnp.zeros(x), output_shape_2)

(Array([0.], dtype=float64), Array([0., 0., 0.], dtype=float64))

# Investigate broadcasting behavior in vmap

In [121]:
n_max_neighbors = 10
n_dim = 1

R_ij_row = jnp.ones((n_max_neighbors, n_dim))

In [122]:
def pair_eval_fun(R):
    return jnp.linalg.norm(R)

# def pair_eval_fun(R):
#     return R / jnp.linalg.norm(R)

pair_eval_fun_output_shape = pair_eval_fun(R_ij_row[0]).shape

cond = jnp.arange(n_max_neighbors) < 4
# TODO: squeeze or not?
# cond_reshaped = jnp.tile(cond, (*pair_eval_fun_output_shape, 1)).T.squeeze()
cond_reshaped = jnp.tile(cond, (*pair_eval_fun_output_shape, 1)).T

pair_eval_fun_output_shape, cond_reshaped.shape, cond_reshaped

((),
 (10,),
 Array([ True,  True,  True,  True, False, False, False, False, False,
        False], dtype=bool))

In [123]:
result_before_where = jax.vmap(pair_eval_fun, in_axes=0)(R_ij_row)

result_before_where.shape, result_before_where

((10,), Array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=float64))

In [124]:
cond_reshaped.shape, result_before_where.shape

((10,), (10,))

In [125]:
result_after_where = jnp.where(cond_reshaped, result_before_where, 0.0)

result_after_where.shape, result_after_where

((10,), Array([1., 1., 1., 1., 0., 0., 0., 0., 0., 0.], dtype=float64))